# Scraping with GCD rerun: pointwise violation

At fixed $\mathbf{x}$, the source enters the local power balance $\overline{I_i}(V_i + x_i - Z_\mathrm{s} I_i)$ linearly and only at its own degree of freedom. The step $\delta\mathbf{V} = -\eta(\mathbf{V}_j + \mathbf{x} - Z_\mathrm{s}\mathbf{I})$ reduces every local violation by the factor $1-\eta$; $\eta = 1$ would give the trivial source $(\mathbf{Z}_0 + Z_\mathrm{s}\mathbf{1})\mathbf{I}$, for which $\mathbf{x}$ is the full-plate current, so $\eta$ is set by the linearity check.

Each step changes the source, accepts the step size $\eta$ by the first-order check of the dual value at fixed projectors, and reruns GCD warm (projectors kept, `NGCD_RE` iterations). Scraping stops when the bound falls below 70 % of $D_0$, when the perturbed problem is strongly dual ($|d(\mathbf{V}_j)|$ and $|c_\mathrm{im}(\mathbf{V}_j)|/r^2$ below `STRONG_TOL`), or after `JMAX` steps. The initial GCD state comes from `scrapingInitGCD.py`; shared code is in `scrapingCommon.py`.

In [ ]:
import os
STRATEGY = 'pointwise'
NGCD_RE = int(os.environ.get('NGCD_RE', 10))      # GCD iterations per scraping step
JMAX = int(os.environ.get('JMAX', 50))            # maximum number of scraping steps
FIT = int(os.environ.get('FIT', 1))               # chained density fit
TARGET = os.environ.get('TARGET', 'own')          # constraints of the own source V_j or of the original V
ETA0 = float(os.environ.get('ETA0', 1.0))
STRONG_TOL = float(os.environ.get('STRONG_TOL', 1e-4))
INIT = os.environ.get('INIT')                     # results/scrapingInitGCD_*.npz, default newest
TAG = os.environ.get('TAG', 'local')
from scrapingCommon import *
Q0, ngcd0 = loadInitialGCD(INIT)
ngcdMax = Q0.n_proj_constr if ngcd0 is None else 2 + 2*ngcd0
print(f'{STRATEGY}: NGCD_RE {NGCD_RE}, JMAX {JMAX}, TARGET {TARGET}, ETA0 {ETA0}, STRONG_TOL {STRONG_TOL}, max projectors {ngcdMax}')

In [ ]:
step = pointwiseStep

In [ ]:
t0 = time.time()
hist = scrapeGCD(Q0, step, NGCD_RE, JMAX, ngcdMax, eta0=ETA0, strongTol=STRONG_TOL)
print(f"{len(hist) - 1} steps, {time.time() - t0:.0f}s, stop: {hist[-1]['stop'] or 'JMAX reached'}")

In [ ]:
%matplotlib inline
fig = plotHistory(hist, f'scraping with GCD rerun: {STRATEGY}', f'results/scrapingGCD_{STRATEGY}_{TAG}.png')

In [ ]:
D0 = hist[0]['D']
print(f"{'j':>3s} {'eta':>8s} {'D/D0':>8s} {'e':>9s} {'d(V)':>10s} {'d(Vj)':>10s} {'cim(V)':>10s} {'cim(Vj)':>10s} {'viol':>9s}")
for h in hist:
    print(f"{h['j']:3d} {h['eta']:8.2e} {h['D']/D0:8.5f} {h['e']:9.2e} {h['d']:+10.2e} {h['dOwn']:+10.2e} {h['cim']:+10.2e} {h['cimOwn']:+10.2e} {h['viol']:9.2e}")

In [ ]:
fitOut = chainedFit(hist) if FIT else None

In [ ]:
saveResults(f'results/scrapingGCD_{STRATEGY}_{TAG}.npz', hist, fitOut, D0=D0, Pfull=Pfull, strategy=STRATEGY,
            target=TARGET, ngcdRe=NGCD_RE, init=str(INIT))